In [ ]:

# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from imblearn.over_sampling import SMOTE


# 2. LOAD DATA

df = pd.read_csv("beans_data.csv")

print(df.head())
print(df.info())
print(df.describe())


# 3. EDA


# Class distribution
plt.figure(figsize=(8,5))
sns.countplot(x='Class', data=df)
plt.xticks(rotation=45)
plt.title("Class Distribution")
plt.show()

# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df.drop('Class', axis=1).corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

# 4. CHECK MISSING VALUES

print("Missing Values:\n", df.isnull().sum())


# 5. FEATURE & TARGET

X = df.drop('Class', axis=1)
y = df['Class']


# 6. TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# 7. SCALING

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# 8. HANDLE IMBALANCE (SMOTE)

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)


# 9. MODELS

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC()
}

results = []


# 10. TRAIN + EVALUATE

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    train_acc = model.score(X_train, y_train)
    test_acc = accuracy_score(y_test, y_pred)
    
    print(f"\n===== {name} =====")
    print("Train Accuracy:", train_acc)
    print("Test Accuracy:", test_acc)
    print(classification_report(y_test, y_pred))
    
    results.append([name, train_acc, test_acc])


# 11. RESULTS TABLE

results_df = pd.DataFrame(results, columns=["Model", "Train Accuracy", "Test Accuracy"])
print("\nModel Comparison:\n", results_df)


# 12. BEST MODEL (Random Forest tuning)

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=3)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("\nBest Parameters:", grid.best_params_)


# 13. FINAL EVALUATION

y_pred = best_model.predict(X_test)

print("\nFinal Model Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Confusion Matrix
plt.figure(figsize=(8,6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()